# Градиентный спуск в физических задачах
Семинар. Кирилл Зорин.

Сегодня четыре задачи из физики, в которых ответ — не число, а форма: путь через реку, профиль горки, расстановка зарядов на шаре, траектория машины на извилистой дороге. Решать их на бумаге не придётся. Каждую мы опишем несколькими числами и дадим градиентному спуску эти числа подбирать.

Но сначала напишем сам спуск — на простой функции, где всё видно глазами.

Положите `physics_lab.py` рядом с ноутбуком (в Colab — через панель «Файлы» слева) и запустите ячейку ниже.

In [1]:
import numpy as np
from physics_lab import River, Hill, Car, charges_start, charges_value_grad
from physics_lab import show_quadratic_step, show_river_trace, show_hill, show_charges
from physics_lab import run_hill_steps, show_hill_learning, show_charge_diagnostics, show_car_comparison
from physics_lab import check_step, check_descent, check_schedule, check_sphere_functions, check_race_goal
from physics_lab import reference_descent

## 1. Один шаг вниз

Возьмём функцию двух переменных

$$F(x,y)=(x-2)^2+16(y+1)^2.$$

Её минимум видно сразу: точка $(2,-1)$, там $F=0$. Но спуск об этом не знает. Он стоит в точке $(-2, 0)$, чувствует только наклон под ногами и делает шаг туда, где вниз круче всего — против градиента.

Напишите две функции:

- `demo_gradient(theta)` — градиент $F$ в точке `theta = (x, y)`, то есть две частные производные;
- `gd_step(theta, grad, lr)` — один шаг: сдвинуться от `theta` на `lr`, умноженный на градиент, в обратную ему сторону.

Прежде чем запускать, прикиньте: из точки $(-2, 0)$ куда сдвинется $x$, куда $y$, и что из них сдвинется сильнее? Ячейка нарисует ваш шаг и, для сравнения, шаг в обратную сторону.

In [6]:
def demo_gradient(theta):
    x, y = theta
    grad_x = 2 * (x - 2)
    grad_y = 32 * (y + 1)
    return np.array([grad_x, grad_y])

def gd_step(theta, grad, lr):
    return theta - lr * grad

In [7]:
start = np.array([-2., 0.])
step_ready = False
try:
    g = demo_gradient(start.copy())
    forward = gd_step(start.copy(), g, 0.025)
    opposite = gd_step(start.copy(), g, -0.025)
    show_quadratic_step(start, forward, opposite)
    step_ready = check_step(demo_gradient, gd_step)
except NotImplementedError as error:
    print(error)

Состояние,"Параметры (x, y)",F
θ₀: старт,"[-2.0, 0.0]",32.00
θ₁: шаг вашего кода,"[-1.8, -0.8]",15.08
θ₂: противоположное направление,"[-2.2, 0.8]",69.48


Производные и один шаг верны.


### Много шагов подряд

Один шаг — ещё не спуск. Ниже цикл, который делает `steps` шагов и запоминает, где точка побывала. В нём одна ошибка: первый шаг выходит правильным, а дальше точка идёт не туда. Найдите и исправьте.

Подумайте: после первого шага уже известно, в какую сторону «вниз». Почему нельзя просто идти туда и дальше?

In [12]:
def gradient_descent(value_grad, start, lr, steps, project=lambda x: x):
    theta = np.array(start, dtype=float).copy()
    history = [theta.copy()]

    for _ in range(steps):
        value, grad = value_grad(theta)
        theta = project(gd_step(theta, grad, lr))
        history.append(theta.copy())
    return np.array(history)

In [13]:
gd_ready = step_ready and check_descent(gradient_descent)
solver = gradient_descent if gd_ready else None

Цикл использует градиент в новой точке.


<details><summary>Если цикл не заводится</summary>

Если время уходит, а цикл не работает, спросите преподавателя: он разрешит раскомментировать строку ниже, и дальше вы пойдёте с готовым спуском. К своему циклу можно вернуться потом.

</details>

In [14]:
# solver = reference_descent
if solver is reference_descent:
    print("Сейчас используется готовый цикл GD.")

## 2. Переправа через реку

Нужно добраться из точки A в точку B, а между ними река шириной 150 м. По берегу вы бежите со скоростью 5 м/с, в воде плывёте 2,8 м/с, и при этом течение сносит вас влево, в сторону от B, со скоростью 1,2 м/с. Как добраться быстрее всего?

![Параметры задачи](attachment:river-parameters.png)

Самый короткий путь — прямая AB. Но в воде вы почти вдвое медленнее, чем на суше, да ещё и сносит. Поэтому можно выиграть, если войти в воду в другом месте и грести не туда, куда смотрит прямая.

Путь задаётся двумя числами `theta = (p, k)`:

- `p` — где вы входите в воду. Это координата вдоль берега, измеренная в ширинах реки: $p = 2$ значит 300 м от левого края карты.
- `k` — куда вы гребёте. На каждый метр, проплытый поперёк реки, вы проплываете ещё $k$ метров вдоль неё, в сторону B (это без учёта течения). При $k = 0$ гребёте строго поперёк, при $k = 1$ — под 45°.

Начинаем с прямой: входим туда, где AB пересекает берег, и гребём вдоль AB. Течение при этом сносит, и из воды вы выходите уже не на прямой.

`river.value_grad(theta)` по паре `(p, k)` считает, сколько секунд займёт весь путь от A до B, и градиент этого времени. Ваш спуск получает только эту функцию — больше он про реку ничего не знает.

В начале у градиента первая координата отрицательная, вторая положительная. Что сделает первый шаг: вы войдёте в воду левее или правее? Будете грести круче поперёк или сильнее наискосок? Ответьте, потом смотрите анимацию: она сначала показывает один шаг, а кнопкой можно прокрутить весь спуск.

In [15]:
river = River(current=-1.2)
river_history = None
if solver is not None:
    river_history = solver(river.value_grad, river.start, 0.01, 300)
    show_river_trace(river, river_history)

Параметр,До шага,Градиент,После первого шага
p: место входа / 150 м,2.375,-1.992,2.395
k: собственный курс,1.717,28.718,1.430


/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Начальный маршрут: 252.74 с. После 300 обновлений: 229.02 с.


## 3. Горка: какая форма самая быстрая

Шарик отпускают без толчка в точке A, и он без трения скатывается по жёлобу в точку B — на 3 м ниже и на 10 м правее. Какой формы сделать жёлоб, чтобы шарик докатился быстрее всего?

Прямой жёлоб — самый короткий. Самый ли он быстрый — сейчас узнаем.

![Параметры задачи](attachment:hill-parameters.png)

Форму жёлоба задают две точки C и D — как «ручки» у кривой в графическом редакторе. Жёлоб выходит из A в направлении C и приходит в B со стороны D, но через сами C и D проходить не обязан. У каждой точки два числа: насколько она правее A и насколько ниже: `theta = (x_C, d_C, x_D, d_D)`.

Спуск двигает C и D так, чтобы время скатывания уменьшалось. Кнопка гонки потом пускает шарики по уже найденным формам.

Ниже спуск запущен три раза из прямого жёлоба, каждый раз 80 шагов, с разным размером шага `lr` (в формулах его пишут как η): 0,03, 3 и 100.

Посмотрите на графики. Какой запуск еле ползёт? Какой быстро находит хорошую форму? А какой дёргается туда-сюда?

In [16]:
hill = Hill(drop=3)
hill_runs = {}
if solver is not None:
    hill_runs = run_hill_steps(solver, hill)
    show_hill_learning(hill, {f"η = {lr}": history for lr, history in hill_runs.items()})

Запуск,"В конце, с"
η = 0.03,2.2561
η = 3,1.8817
η = 100,1.9620


### Шаг, который уменьшается

Маленький шаг ползёт, большой скачет вокруг хорошей формы и не может в неё попасть. Попробуем взять лучшее от обоих: начать с большого шага и уменьшать его со временем. На шаге номер $t$ будем брать

$$\eta_t=\frac{\eta_0}{1+\text{decay}\cdot t},\qquad t=0,1,2,\ldots$$

При $t = 0$ это просто $\eta_0$, а при `decay = 0.1` к десятому шагу шаг уменьшится вдвое.

В ячейке ниже допишите одну строку — вычисление `eta`. Ячейка после неё сравнит ваш спуск с постоянным шагом 100 из того же старта.

Подумайте: градиент же показывает, куда вниз. Как шаг в правильную сторону может сделать хуже?

In [21]:
def decayed_descent(value_grad, start, lr0, decay, steps, project=lambda x: x):
    theta = np.array(start, dtype=float).copy()
    history = [theta.copy()]
    for t in range(steps):
        value, grad = value_grad(theta)
        eta = lr0 / (1+decay*t)
        theta = project(theta - eta*grad)
        history.append(theta.copy())
    return np.array(history)

In [22]:
decay_ready = check_schedule(decayed_descent)
hill_fast = None
if decay_ready:
    hill_fast = decayed_descent(hill.value_grad, hill.start, 100, 0.1, 80, hill.project)
    if hill_runs:
        show_hill_learning(hill, {"Постоянный η = 100": hill_runs[100], "Уменьшающийся η": hill_fast})
    show_hill(hill, hill_fast)

Шаг уменьшается по заданному правилу.


Запуск,"В конце, с"
Постоянный η = 100,1.9620
Уменьшающийся η,1.8795


/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Прямой спуск: 2.722 с. Ваша горка: 1.879 с.


## 4. Заряды на сфере

На поверхности шара сидят 24 одинаковых заряда — как бусины, которые могут ездить по сфере, но не могут с неё слететь. Заряды отталкиваются и расползаются, пока не встанут так, что энергия отталкивания станет минимальной. Как они встанут?

Для двух зарядов ответ очевиден — на противоположных полюсах. Для четырёх — в вершинах тетраэдра. Для 24 угадать уже трудно.

Энергия — это сумма $1/r$ по всем парам зарядов: чем ближе пара, тем больше её вклад.

![Параметры задачи](attachment:charges-24-start.png)

Чисел, которые подбирает спуск, тут много: `theta` — таблица 24 × 3, в каждой строке координаты $(x, y, z)$ одного заряда. Спуску всё равно, сколько их.

Запустим тот же спуск, что и раньше, ничего не меняя.

In [23]:
initial_charges = charges_start(24, seed=1)
free_history = None
if solver is not None:
    free_history = solver(charges_value_grad, initial_charges, 0.25/24, 50)
    show_charge_diagnostics(initial_charges, free_history[-1])

Состояние,Энергия,Минимальный радиус,Максимальный радиус
Старт,257.931,1.000,1.000
Без ограничения,89.017,2.304,2.782


**Посмотрите на таблицу: что случилось с зарядами?** Ответьте для себя, прежде чем читать дальше.

Спуск не знает, что заряды обязаны оставаться на сфере. Он видит только энергию, а её проще всего уменьшить, если разлететься подальше друг от друга. Поэтому после каждого шага будем возвращать заряды на место: каждую точку двигаем вдоль её радиуса, пока она не окажется на сфере. Это деление вектора на его длину:

$$p_i\leftarrow \frac{p_i}{\|p_i\|}.$$

Делается для каждой строки по отдельности.

Напишите:

- `on_sphere(points)` — правда ли, что все точки лежат на сфере, то есть длина каждой строки равна 1 с точностью `1e-6`;
- `project_sphere(points)` — вернуть каждую точку на сферу.

Следующая ячейка запустит спуск заново и после каждого шага будет вызывать ваш `project_sphere`.

In [25]:
def on_sphere(points):
    lengths = np.sqrt(points[:, 0]**2 + points[:, 1]**2 + points[:, 2]**2)
    return np.all(np.abs(lengths - 1.0) < 1e-6)

def project_sphere(points):
    lengths = np.sqrt(points[:, 0]**2 + points[:, 1]**2 + points[:, 2]**2)
    return points / lengths[:, None]

In [26]:
sphere_ready = check_sphere_functions(on_sphere, project_sphere)
charge_history = None
if solver is not None and sphere_ready:
    charge_history = solver(charges_value_grad, initial_charges, 0.25/24, 300, project_sphere)
    print("Все точки на сфере:", on_sphere(charge_history[-1]))
    show_charges(charge_history)

Проверка радиуса и возврат на сферу работают.
Все точки на сфере: True


/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Энергия: 257.931293 → 223.354880. Максимальная ошибка радиуса: 3.3e-16.


## 5. Машина на извилистой дороге

Дорога — змейка из двух поворотов длиной 120 м и шириной 14 м. Машина стоит на старте и должна остановиться точно на финише. Разгоняется она не быстрее 3 м/с², тормозит не сильнее 5 м/с², а в повороте её держит только сцепление: чем круче дуга, тем медленнее по ней можно ехать, иначе вынесет. Где ехать по ширине дороги, чтобы доехать быстрее всего?

![Параметры задачи](attachment:car.png)

Траекторию задают семь чисел: насколько машина отклоняется от середины дороги в семи точках вдоль неё, в одну или в другую сторону, в метрах. Между этими точками линия сглаживается. Семь нулей — это езда ровно по центру.

У `car` две функции, как у реки: `car.length_value_grad(theta)` считает длину пути и её градиент, `car.value_grad(theta)` — время заезда и его градиент.

Спуск ниже уже запущен, но он ищет самую короткую траекторию, а спрашивают нас про самую быструю. Исправьте `race_objective`, чтобы спуск искал то, что нужно, и сравните два маршрута.

In [30]:
car = Car(width=5)
car_short = None
if solver is not None:
    car_short = solver(car.length_value_grad, car.start, 2, 250, car.project)

In [32]:
def race_objective(theta, scene):
    # Сейчас спуск ищет самый короткий путь. Нужен самый быстрый.
    return scene.value_grad(theta)

In [33]:
race_ready = check_race_goal(race_objective, car)
car_fast = None
if solver is not None and race_ready and car_short is not None:
    car_fast = solver(lambda theta: race_objective(theta, car), car.start, 1, 250, car.project)
    show_car_comparison(car, car_short, car_fast)

Цель и её градиент соответствуют времени поездки.


Цель запуска,"Длина, м","Время, с"
Длина,143.80,13.919
Время,146.66,13.606


/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Минимизация длины: 13.919 с. Ваш маршрут: 13.606 с.


Посмотрите гонку и график скорости. Быстрая траектория длиннее — где она отыгрывает время?

Если останется время: сделайте дорогу уже (`Car(width=...)`). Прежде чем запускать, предскажите, что станет с быстрой траекторией.

## Что мы сегодня видели

- В первой задаче шаг против градиента уменьшил $F$, а такой же шаг по градиенту — увеличил. Почему для маленького шага это так всегда?
- Почему в цикле градиент приходится считать заново на каждом шаге?
- На горке шаг 100 оказался хуже, чем шаг 3, хотя оба шли «вниз». Как так вышло?
- Заряды без поправки получили энергию гораздо меньше, чем с поправкой. Почему это не лучший ответ? И почему самый короткий маршрут машины — не лучший?